# Replication

This notebook documents Google Drive replication support for `@docstack/pouchdb-adapter-googledrive`: the automated regression test tool that exercises it against a real Drive account, the bugs that tool has found and fixed, and the historical debugging/root-cause record from earlier replication issues (including a translated archive of the original diagnostic session).

**Sections:**
1. Production Replication Round-Trip (test tool)
2. Bugs found by this tool
3. Root cause analysis: earlier replication fixes (merged from `REPLICATION_ISSUE.md`)
4. Archived debug session: bidirectional replication (translated from `DEBUG_BIDIRECTIONAL_REPLICATION.ipynb`, Italian -> English)

> This notebook is a documentation artifact, not a runnable one. Code cells here are reference snippets (a mix of TypeScript/JS and bash) kept as code cells for syntax highlighting and readability - they are not wired up to execute against this repo's test infrastructure or kernel.

## 1. Production Replication Round-Trip (test tool)

`tests/production.replication.test.ts` verifies PouchDB's standard replication protocol works correctly *through* a real Google Drive-backed database, exercising both a no-op idempotency leg and a genuine bidirectional-writes leg - the real-world shape of two devices syncing through a shared Drive folder.

```
classical (memory) instance A --replicate--> Google Drive instance B       (Phase 2)
Google Drive instance B       --replicate--> classical (memory) instance C (Phase 3)
classical (memory) instance C --replicate--> classical (memory) instance A (Phase 4, idempotency check)
... C makes its own writes: updates to docs that originated on A, plus brand new docs ...
classical (memory) instance C --replicate--> Google Drive instance B       (Phase 6)
Google Drive instance B       --replicate--> classical (memory) instance A (Phase 7)
```

It writes documents and a couple of updates (second revisions) on A, replicates A -> B, deep-compares every document (including exact revision IDs, not just content) on B against A, replicates B -> C and compares the same way, then replicates C -> A to close the loop as a no-op (idempotency check: PouchDB's checkpoint/revsDiff machinery should find nothing new to write, and A must come out byte-identical).

It then has C make real local writes - updates to two of the A-originated docs, plus two brand new documents - pushes those into the Drive hub (C -> B), and pulls the merged state back down to A (B -> A), confirming A ends up matching C exactly and every untouched document survived the whole session byte-identical.

In [ ]:
# Running it
npm run test:prod:replication

# Windows: npm scripts run through cmd.exe, which doesn't understand `VAR=value cmd`.
# Run through Git Bash instead:
TEST_ENV=production npx jest tests/production.replication.test.ts

# Env vars:
#   KEEP_TEST_DATA=1        skip deleting the Drive folder at the end
#   PROD_TEST_DOC_COUNT=10  override how many documents to write on instance A

### A test-harness gotcha worth knowing: `activeTasks`

The very first version of this test hung forever with no error, no matter how long the timeout. Root cause turned out to be nothing to do with the Google Drive adapter: pouchdb-replication's internal `createTask()`/`completeReplication()` call `src.activeTasks.add/remove/update` unconditionally on whichever database is acting as the replication *source*. This adapter provides `activeTasks` for itself (see `src/adapter.ts`), but a plain `pouchdb-core` + `pouchdb-adapter-memory` instance (used here as the "classical" comparison instances) does not - `pouchdb-adapter-memory` apparently expects the full `pouchdb` package to supply it. So the first `memory -> gdrive` replication crashed deep inside pouchdb-replication with `Cannot read properties of undefined (reading 'add')`. Because that throw happens inside an internal, un-awaited promise chain, it never reaches the caller's `'error'` listener - from the outside, replication just hangs forever instead of failing loudly. The test works around this with a small polyfill (`ensureActiveTasks()`) applied to the memory-adapter instances; it's a gap in this bare-`pouchdb-core` test setup, not a bug in `src/`, so no adapter code changed for it.

## 2. Bugs found by this tool

### `db.bulkDocs(..., { new_edits: false })` silently discarded replicated revisions

The most serious finding to date: replicating data *into* a Drive-backed database never actually preserved revision history. Every replicated document kept its correct content but was assigned a brand-new, fabricated revision instead of the one it arrived with - so `doc-0`'s real revision `2-258a45c9...` (with a correct 2-entry `_revisions.ids` ancestor chain) got silently replaced with something like `1-p6den3crcjdl8vns63` (a random hash at the wrong generation, with a corrupted 3-entry ancestor chain). Any later two-way sync with the original source would see these as a hard conflict, because the revision trees no longer agree on history - despite the content being identical.

**Root cause:** `api.bulkDocs = api._bulkDocs` (in `src/adapter.ts`) aliases the public method straight to the adapter's own implementation, bypassing `AbstractPouchDB.prototype.bulkDocs` - the PouchDB core wrapper that normally copies `req.new_edits` onto `opts.new_edits` before calling the adapter (see `pouchdb-core/lib/index.js`). Because that normalization never ran, and `_bulkDocs` read `opts.new_edits !== false` instead of `req.new_edits !== false`, every replication write - which pouchdb-replication issues as `target.bulkDocs({ docs, new_edits: false }, { timeout })`, i.e. with `new_edits` on `req`, not `opts` - looked exactly like a normal, brand-new edit. The adapter then ran its default "mint a new revision on top of whatever's currently in the index" logic instead of trusting the incoming revision, exactly as if a first-time author had written the doc rather than a replicator delivering an existing one.

This one was hard to see from the outside: `allDocs`/`get` still returned the *correct document content* (so a casual smoke test would pass), only `_rev`/`_revisions` were wrong. It only became visible by comparing revision IDs across the replication hop, which is exactly what this tool's `compareDocSets()` does (deliberately, not just comparing content) - a plain content-only diff would have missed it entirely. **Fixed** by reading `new_edits` from `req` instead of `opts`.

This bug is also a flag that **every** `api.X = api._X` alias in this file (`get`, `allDocs`, `bulkGet`, `destroy`, ...) bypasses its corresponding `AbstractPouchDB.prototype.X` wrapper the same way `bulkDocs` did - worth keeping in mind if another PouchDB-core-level request/options normalization step turns out to matter for one of those in the future.

### `_local/*` documents leaked into `allDocs()`

Replication checkpoint documents (`_local/<hash>`) were never excluded from `_allDocs`'s candidate key set, so they showed up in `allDocs()` results - real CouchDB/PouchDB never surfaces `_local/*` docs there, they live in a separate namespace entirely. This is a sibling of the deleted-docs-leaking-into-`allDocs()` bug found by the Production Drive Explorer tool (see `docs/TESTING.md`) - same function, different gap. Fixed alongside it.

## 3. Root cause analysis: earlier replication fixes

*(Merged from the former `docs/REPLICATION_ISSUE.md`.)*

After extensive debugging with a live emulation test suite, multiple critical issues were identified in the `_changes` and `notifyListeners` implementations.

#### 3.1 Missing `seq` Property in Notifications (CRITICAL)
**Location:** `src/drive.ts` -> `notifyListeners()`

**Problem:** The `notifyListeners` method was emitting change objects that included `id`, `rev`, and `deleted`, but omitted the `seq` (sequence number).

**Consequence:** PouchDB's replication engine uses the `seq` to track checkpoints. If a change arrives without a `seq`, PouchDB silently ignores it or fails to update its "since" pointer, meaning the replication appears to "detect" a change but never actually processes the document.

**Fix:** Updated the notification object to include the correct sequence number from the index.

---

#### 3.2 Infinite Reconnect Loop in Live Changes
**Location:** `src/adapter.ts` -> `_changes()`

**Problem:** An initial "fix" (based on standard CouchDB behavior) suggested calling `opts.complete()` at the end of the initial changes batch, even for live feeds.

**Consequence:** In PouchDB, calling `complete()` on a **live** stream signals that the stream has ended gracefully. Because the replication is set to `live: true`, PouchDB interprets this as a connection drop and immediately restarts the changes feed. This created an infinite loop of:
`Connect -> Fetch Batch -> Call Complete -> Disconnect -> Reconnect`
This loop caused high CPU usage and eventually a "JavaScript heap out of memory" crash.

**Fix:** Restricted `opts.complete()` to only be called if `!opts.live`.

---

#### 3.3 System Document Leakage (`_local/`)
**Location:** `src/adapter.ts` -> `_changes()`

**Problem:** PouchDB uses local documents (IDs starting with `_local/`) to store replication checkpoints. The adapter was incorrectly including these internal documents in the `_changes` feed results.

**Consequence:** When the remote side (Google Drive) reported a change to a `_local/` document, the local PouchDB tried to fetch it to replicate it. However, local documents are non-replicable and often missing from the full document fetch logic, leading to `not_found: missing` errors that crashed the replication process.

**Fix:** Added a filter to both the initial batch and the live listener to ignore any document ID starting with `_local/`.

---

#### 3.4 Inefficient Polling & Redundant `load()` Calls
**Location:** `src/drive.ts` -> `startPolling()` / `load()`

**Problem:** The polling mechanism was frequently triggering `load()`, which cleared the entire `processedLogIds` cache and re-downloaded all change logs from Google Drive. Additionally, `load()` was being called on every conflict retry without checking if the metadata had actually changed.

**Fix:**
1. Implemented **ETag-based change detection** in polling.
2. Added an **ETag check at the start of `load()`** to skip processing if the metadata on the server hasn't changed since the last load.
3. Updated `tryAppendChanges` to proactively call `notifyListeners()` for local writes, reducing reliance on the next polling tick for local UI updates.

**Summary of results:** With these fixes applied, bidirectional replication became stable and efficient. The "Out of Memory" crashes were resolved, and changes from Google Drive were correctly detected and propagated to the local database within the specified polling interval.

## 4. Archived debug session: bidirectional replication

*(Translated to English from the former `docs/DEBUG_BIDIRECTIONAL_REPLICATION.ipynb`, an Italian-language diagnostic notebook. This is the debugging session that originally surfaced the symptoms behind the root causes in Section 3 above. Preserved here as a historical record - the code cells reference an external app's variables (`stack?.db`, `driveConfig`) and are not runnable against this repo's test suite as-is; per instruction, this archived material's logic has not been wired up.)*

### Problem
Replication `localDB.replicate.to(googleDriveDB)` works (OK)
But `localDB.replicate.from(googleDriveDB)` does not sync data (BROKEN)

This notebook diagnoses the replication flow to identify where the block occurs.

### 4.1 Setup and Replication Configuration

Add this code to your component to diagnose the problem:

In [ ]:
// Setup Google Drive Adapter with debug logging
const plugin = GoogleDriveAdapter({
    accessToken: async () => driveConfig.accessToken,
    folderName: 'my-db',
    pollingIntervalMs: 5000,
    debug: true  // Enable detailed logging
});

PouchDB.plugin(plugin);

const googleDriveDB = new PouchDB('paper-drive-db', {
    adapter: 'googledrive'
});

const localDB = stack?.db;

// Add listeners to monitor replication events
console.log('=== BIDIRECTIONAL REPLICATION SETUP ===');

// Replicate TO (localDB -> googleDriveDB) - Working
const repTo = localDB.replicate.to(googleDriveDB, { live: true, retry: true });
repTo.on('change', (info) => {
    console.log('TO: change event', {
        direction: 'local->gdrive',
        docs: info.docs?.length,
        ok: info.ok
    });
});
repTo.on('complete', (info) => {
    console.log('TO: replication complete', info);
});
repTo.on('error', (err) => {
    console.error('TO: replication error', err);
});

// Replicate FROM (googleDriveDB -> localDB) - Problem
const repFrom = localDB.replicate.from(googleDriveDB, { live: true, retry: true });
repFrom.on('change', (info) => {
    console.log('FROM: change event', {
        direction: 'gdrive->local',
        docs: info.docs?.length,
        ok: info.ok
    });
});
repFrom.on('complete', (info) => {
    console.log('FROM: replication complete', info);
});
repFrom.on('error', (err) => {
    console.error('FROM: replication error', err);
});

### 4.2 Diagnosis - Checking the Replication Flow

Check the sequences and metadata of both databases

In [ ]:
// Compare the state of both databases
async function compareDatabases() {
    const localInfo = await localDB.info();
    const googleDriveInfo = await googleDriveDB.info();

    console.log('DATABASE INFO COMPARISON:');
    console.log('Local DB:', {
        doc_count: localInfo.doc_count,
        update_seq: localInfo.update_seq,
        db_name: localInfo.db_name
    });
    console.log('Google Drive DB:', {
        doc_count: googleDriveInfo.doc_count,
        update_seq: googleDriveInfo.update_seq,
        db_name: googleDriveInfo.db_name
    });

    return { localInfo, googleDriveInfo };
}

// Run the diagnosis after 25 seconds (giving polling time to detect changes)
setTimeout(async () => {
    console.log('\n=== DIAGNOSIS AFTER 25 SEC ===\n');

    const { localInfo, googleDriveInfo } = await compareDatabases();

    // Check whether the sequences are in sync
    if (localInfo.update_seq === googleDriveInfo.update_seq) {
        console.log('Sequences in sync');
    } else {
        console.warn('Sequences NOT in sync:', {
            local: localInfo.update_seq,
            googleDrive: googleDriveInfo.update_seq,
            diff: googleDriveInfo.update_seq - localInfo.update_seq
        });
    }

    // Check the specific document on Google Drive
    try {
        const docOnDrive = await googleDriveDB.get('Notebook-0');
        console.log('Notebook-0 found on Google Drive:', {
            _id: docOnDrive._id,
            _rev: docOnDrive._rev
        });
    } catch (err) {
        console.error('Notebook-0 NOT found on Google Drive:', err.message);
    }

    // Check the document on the Local DB
    try {
        const docLocal = await localDB.get('Notebook-0');
        console.log('Notebook-0 found on Local DB:', {
            _id: docLocal._id,
            _rev: docLocal._rev
        });
    } catch (err) {
        console.error('Notebook-0 NOT found on Local DB:', err.message);
    }

}, 25000);

### 4.3 Debug - Polling and Change Detection

The problem is likely in the polling mechanism.

Check whether polling is actually detecting changes from Google Drive:

In [ ]:
// Monitor changes detected via the _changes API
async function debugChangesAPI() {
    console.log('\n=== TESTING _CHANGES API ===\n');

    // Get the last 10 changes from the Google Drive DB
    try {
        const changes = await googleDriveDB.changes({
            include_docs: true,
            descending: true,
            limit: 10
        });

        console.log('Recent changes from Google Drive DB:', {
            results: changes.results.length,
            last_seq: changes.last_seq,
            results: changes.results.map(r => ({
                id: r.id,
                seq: r.seq,
                changes: r.changes,
                deleted: r.deleted,
                hasDoc: !!r.doc
            }))
        });

        return changes;
    } catch (err) {
        console.error('Error fetching changes from Google Drive DB:', err);
    }
}

// Test the _changes API immediately
await debugChangesAPI();

// Test again after 10 seconds to see if polling detected new changes
setTimeout(async () => {
    console.log('\n=== TESTING _CHANGES API AFTER POLLING ===\n');
    await debugChangesAPI();
}, 10000);

## Possible culprits identified

### Issue 1: `_changes` feed with `include_docs` might be slow

In `adapter.ts`, when `opts.include_docs` is true, the `_changes` method has to:
- Download the BODY of every document (slow)
- This blocks replication

**Diagnosis:** Check the browser console. If you see many `fetchFile` calls for every change, that's the bottleneck.

### Issue 2: Change detection doesn't respect the `since` parameter

PouchDB replication sends `since: lastSeq` to receive only NEW changes.
If `_changes` doesn't filter correctly, replication won't detect new documents.

### Issue 3: The live listener might not be called

In `DriveHandler`, polling detects changes to `_meta.json`, but:
- Polling has a 5-second delay
- Replication might time out before receiving notifications

### 4.4 Troubleshooting - Revision Conflicts

Check revision validation

In [ ]:
// Check for revision conflicts during replication
async function checkReplicationConflicts() {
    console.log('\n=== CHECKING REPLICATION CONFLICTS ===\n');

    // Read the same document from both databases
    const docId = 'Notebook-0';

    try {
        const localDoc = await localDB.get(docId);
        console.log('Local Doc:', { _id: localDoc._id, _rev: localDoc._rev });
    } catch (e) {
        console.warn('Local doc not found');
    }

    try {
        const remoteDoc = await googleDriveDB.get(docId);
        console.log('Remote Doc:', { _id: remoteDoc._id, _rev: remoteDoc._rev });
    } catch (e) {
        console.warn('Remote doc not found');
    }

    // Check _conflicts (documents with merge conflicts)
    try {
        const allDocs = await localDB.allDocs({ conflicts: true });
        const conflictedDocs = allDocs.rows.filter(r => r.value.conflicts && r.value.conflicts.length > 0);
        if (conflictedDocs.length > 0) {
            console.warn('Found documents with conflicts:', conflictedDocs);
        } else {
            console.log('No conflicts found in local DB');
        }
    } catch (e) {
        console.error('Error checking conflicts:', e);
    }
}

// Run the check after 30 seconds
setTimeout(async () => {
    await checkReplicationConflicts();
}, 30000);

### 4.5 Validation - Bidirectional Replication Test

Manual test: create a document on Google Drive and verify it syncs

In [ ]:
// Full Bidirectional Replication Test

async function runFullReplicationTest() {
    console.log('\n=== FULL REPLICATION TEST ===\n');

    // Step 1: Read the initial state
    const localInfoBefore = await localDB.info();
    const googleDriveInfoBefore = await googleDriveDB.info();

    console.log('Initial State:');
    console.log('  Local:', { doc_count: localInfoBefore.doc_count, seq: localInfoBefore.update_seq });
    console.log('  Google Drive:', { doc_count: googleDriveInfoBefore.doc_count, seq: googleDriveInfoBefore.update_seq });

    // Step 2: Create a new document ON GOOGLE DRIVE
    const testDocId = `test-sync-${Date.now()}`;
    const testDoc = {
        _id: testDocId,
        title: 'Test Document for Replication',
        createdAt: new Date().toISOString(),
        source: 'google-drive-db'
    };

    try {
        const saveResult = await googleDriveDB.put(testDoc);
        console.log('\nCreated document on Google Drive:', { id: saveResult.id, rev: saveResult.rev });
    } catch (err) {
        console.error('Failed to create document on Google Drive:', err);
        return;
    }

    // Step 3: Wait for polling + replication (20 seconds should be enough)
    console.log('\nWaiting for polling and replication (20 seconds)...');

    await new Promise(resolve => setTimeout(resolve, 20000));

    // Step 4: Check whether the document arrived on the Local DB
    try {
        const docOnLocal = await localDB.get(testDocId);
        console.log('\nSUCCESS! Document replicated to local DB:', {
            id: docOnLocal._id,
            rev: docOnLocal._rev,
            title: docOnLocal.title
        });
    } catch (err) {
        console.error('\nFAILED! Document NOT found on local DB:', err.message);

        // Additional diagnostics
        console.log('\nDIAGNOSTIC INFO:');
        const localInfoAfter = await localDB.info();
        const googleDriveInfoAfter = await googleDriveDB.info();

        console.log('After Test:');
        console.log('  Local:', { doc_count: localInfoAfter.doc_count, seq: localInfoAfter.update_seq });
        console.log('  Google Drive:', { doc_count: googleDriveInfoAfter.doc_count, seq: googleDriveInfoAfter.update_seq });

        // Verify the doc is actually on Google Drive
        try {
            const verifyDoc = await googleDriveDB.get(testDocId);
            console.log('Document IS on Google Drive:', {
                id: verifyDoc._id,
                rev: verifyDoc._rev
            });
            console.log('BUT NOT replicated to local. Issue: REPLICATION FROM REMOTE NOT WORKING');
        } catch (e) {
            console.error('Document not even on Google Drive. Save failed?');
        }
    }
}

// Run the test
await runFullReplicationTest();

---

## Recommended solutions

Based on the results of the test above, here are the steps to resolve the problem:

### If the test fails (document doesn't replicate from remote):

**Solution 1: Increase `pollingIntervalMs`**
```javascript
pollingIntervalMs: 2000  // Instead of 5000 - poll more frequently
```

**Solution 2: Reduce the replication timeout**
```javascript
const repFrom = localDB.replicate.from(googleDriveDB, {
    live: true,
    retry: true,
    timeout: 15000  // Increase timeout from the 30s default
});
```

**Solution 3: Add logging to `adapter._changes` in `adapter.ts`**

Modify `src/adapter.ts` around line 428 to add debug output:
```typescript
api._changes = function (opts: any): { cancel: () => void } {
    console.log('[_changes] Called with opts:', {
        since: opts.since,
        limit: opts.limit,
        live: opts.live,
        include_docs: !!opts.include_docs
    });
    // ...
```

**Solution 4: Verify the live listener is configured correctly**

In `DriveHandler.load()`, make sure polling calls `notifyListeners()` every time it detects a change:
```typescript
if (metaFile.modifiedTime !== this.metaModifiedTime) {
    this.log('Polling detected change!', metaFile.modifiedTime);
    await this.load();
    this.notifyListeners();  // <-- MUST BE CALLED
}
```

## Status

Every root cause identified in this debug session (Section 4) was addressed by the fixes in Section 3, and regression coverage for the resulting bidirectional replication behavior now lives in the automated Production Replication Round-Trip test (Section 1) - which the Section 2 findings came from.